# Day 31 Capstone: Creative Studio, Generate & Understand Images

**Models:** Stable Diffusion v1.5, BLIP-base, LAION CLIP ViT-B/32. All inference runs locally on CPU or CUDA; no hosted inference API or external image dataset. First use downloads weights from Hugging Face Hub. **Note:** CPU Stable Diffusion v1.5 can take much longer than a few minutes, even at 12 steps. Models require several GB of disk and memory.

**Reproducibility:** fixed seed, device, step counts. Outputs depend on library versions and hardware. This notebook saves all artifacts to its current working directory. Run top-to-bottom in JupyterLab after installing dependencies.

In [ ]:
import os, gc, warnings
import numpy as np
import torch
from PIL import Image, PngImagePlugin
from IPython.display import display
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
SEED = 42
STEPS = 12 if DEVICE == 'cpu' else 30
SIZE = 512
print(f'device={DEVICE}, dtype={DTYPE}, seed={SEED}, steps={STEPS}')
def seeded_generator():
    return torch.Generator(device=DEVICE).manual_seed(SEED)
def make_grid(images, cols=3):
    w,h=images[0].size
    grid=Image.new('RGB',(cols*w, ((len(images)+cols-1)//cols)*h),'white')
    for i,image in enumerate(images):grid.paste(image,((i%cols)*w,(i//cols)*h))
    return grid


## 1. Text-to-image generation

Use 512×512 resolution and a fixed seed. The model has a use-based license; follow its license and safety requirements. The safety checker remains enabled.

In [ ]:
from diffusers import StableDiffusionPipeline
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)
prompt='a cozy reading nook by a rainy window, warm lamp light, watercolor style'
with torch.inference_mode():
    img=pipe(prompt, num_inference_steps=STEPS, guidance_scale=7.5, generator=seeded_generator(), height=SIZE, width=SIZE).images[0]
img.save('gen_baseline.png')
print('Saved gen_baseline.png', img.size)
display(img)


## 2. Prompt-engineering experiment

Same seed and prompt, three settings. Guidance can change alignment and contrast; effects vary, and a negative prompt does not guarantee removal of unwanted features.

In [ ]:
variants=[
    ('guidance=3',dict(guidance_scale=3.0)),
    ('guidance=12',dict(guidance_scale=12.0)),
    ('negative prompt',dict(guidance_scale=7.5,negative_prompt='blurry, dark, lowres, deformed')),
]
variant_images=[]
for label,settings in variants:
    with torch.inference_mode():
        im=pipe(prompt, num_inference_steps=STEPS, generator=seeded_generator(), height=SIZE, width=SIZE, **settings).images[0]
    variant_images.append(im)
    print('Generated',label)
grid=make_grid(variant_images)
grid.save('gen_prompt_grid.png')
display(grid)


**Comparison notes (complete after observing your images):** Compare the low-guidance composition with the high-guidance composition, and inspect whether the negative prompt actually reduces blur/darkness. My provisional shipping choice is guidance 7.5 with the negative prompt because it balances alignment and flexibility, but this must be confirmed visually on the generated panels.

## 3. Image-to-image editing

The input image is the baseline. Lower strength usually preserves more structure; higher strength permits larger changes.

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
img2img=StableDiffusionImg2ImgPipeline(**pipe.components).to(DEVICE)
img2img.set_progress_bar_config(disable=True)
init=Image.open('gen_baseline.png').convert('RGB').resize((SIZE,SIZE))
edit_prompt='the same cozy reading nook, but at golden sunset, autumn colors'
edits=[]
for strength in (0.35,0.65):
    with torch.inference_mode():
        edited=img2img(prompt=edit_prompt,image=init,strength=strength,guidance_scale=7.5,num_inference_steps=STEPS,generator=seeded_generator()).images[0]
    edits.append(edited)
    print('Completed img2img',strength)
edit_grid=make_grid([init]+edits)
edit_grid.save('gen_img2img.png')
display(edit_grid)
del img2img, pipe
gc.collect()
if DEVICE=='cuda':torch.cuda.empty_cache()


## 4. Image understanding: BLIP captions and CLIP zero-shot classification

CLIP softmax values are **relative scores among supplied candidate labels**, not calibrated real-world probabilities.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPProcessor, CLIPModel
BLIP_ID='Salesforce/blip-image-captioning-base'
CLIP_ID='laion/CLIP-ViT-B-32-laion2B-s34B-b79K'
blip_proc=BlipProcessor.from_pretrained(BLIP_ID)
blip_model=BlipForConditionalGeneration.from_pretrained(BLIP_ID).to(DEVICE).eval()
scene=Image.open('gen_baseline.png').convert('RGB')
blip_inputs=blip_proc(scene,return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    generated=blip_model.generate(**blip_inputs,max_new_tokens=30)
caption=blip_proc.decode(generated[0],skip_special_tokens=True)
print('BLIP caption:',caption)
del blip_model, blip_inputs
gc.collect()
if DEVICE=='cuda':torch.cuda.empty_cache()
clip_model=CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_proc=CLIPProcessor.from_pretrained(CLIP_ID)
labels=['a cozy indoor scene','a busy city street','a watercolor painting','a photograph of food','an outdoor landscape']
inputs=clip_proc(text=labels,images=scene,return_tensors='pt',padding=True).to(DEVICE)
with torch.inference_mode():
    probabilities=clip_model(**inputs).logits_per_image.softmax(dim=1)[0].cpu().tolist()
for label,prob in sorted(zip(labels,probabilities),key=lambda item:-item[1]):
    print(f'{prob:5.1%} {label}')


## 5. CLIP text-to-image search over self-generated gallery

Reload the diffusion pipeline to minimize peak memory use when both SD and CLIP are needed. Then embed the gallery and rank by normalized cosine similarity.

In [ ]:
pipe=StableDiffusionPipeline.from_pretrained(MODEL_ID,torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)
gallery_prompts=['a red bicycle leaning on a brick wall','a bowl of ramen with steam rising','a snowy mountain peak at dawn','a cozy reading nook with a lamp']
gallery=[]
for i,p in enumerate(gallery_prompts):
    with torch.inference_mode():
        image=pipe(p,num_inference_steps=STEPS,guidance_scale=7.5,generator=seeded_generator(),height=SIZE,width=SIZE).images[0]
    image.save(f'gallery_{i+1}.png')
    gallery.append(image)
print('Generated',len(gallery),'gallery images')
display(make_grid(gallery,cols=2))
del pipe
gc.collect()
if DEVICE=='cuda':torch.cuda.empty_cache()


In [ ]:
import torch.nn.functional as F
# CLIP model may reside on GPU; keep batch small for CPU/GPU compatibility.
def clip_image_embeds(images):
    vectors=[]
    for image in images:
        batch=clip_proc(images=image,return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            vec=clip_model.get_image_features(**batch)
            if not isinstance(vec,torch.Tensor):vec=vec.pooler_output
        vectors.append(F.normalize(vec,dim=-1))
    return torch.cat(vectors,dim=0)
def clip_text_embed(text):
    batch=clip_proc(text=[text],return_tensors='pt',padding=True).to(DEVICE)
    with torch.inference_mode():
        vec=clip_model.get_text_features(**batch)
        if not isinstance(vec,torch.Tensor):vec=vec.pooler_output
    return F.normalize(vec,dim=-1)
img_embeds=clip_image_embeds(gallery)
query='somewhere warm to sit and read'
scores=(clip_text_embed(query) @ img_embeds.T)[0]
print('Query:',query)
for index in scores.argsort(descending=True).tolist():
    print(f'{scores[index].item():.3f} <- {gallery_prompts[index]}')


## 6. Responsible AI: provenance and bias probe

PNG metadata is not a robust watermark: it can be stripped. The occupation probe is an illustration, **not evidence of population-level bias from one sample**.

In [ ]:
meta=PngImagePlugin.PngInfo()
meta.add_text('Generator','stable-diffusion-v1-5')
meta.add_text('AI-Generated','true')
meta.add_text('Seed',str(SEED))
Image.open('gen_baseline.png').save('gen_baseline_tagged.png',pnginfo=meta)
assert Image.open('gen_baseline_tagged.png').info.get('AI-Generated')=='true'
print('Saved tagged PNG with provenance metadata')
pipe=StableDiffusionPipeline.from_pretrained(MODEL_ID,torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)
with torch.inference_mode():
    role_img=pipe('a photo of a nurse',num_inference_steps=STEPS,generator=seeded_generator(),height=SIZE,width=SIZE).images[0]
role_img.save('gen_nurse_probe.png')
del pipe
gc.collect()
if DEVICE=='cuda':torch.cuda.empty_cache()
probe=['a photo of a woman','a photo of a man']
probe_inputs=clip_proc(text=probe,images=role_img,return_tensors='pt',padding=True).to(DEVICE)
with torch.inference_mode():
    pp=clip_model(**probe_inputs).logits_per_image.softmax(dim=1)[0].cpu().tolist()
print('CLIP relative similarity on this generated nurse image:',dict(zip(probe,[round(x,3) for x in pp])))
print('Interpret cautiously: one seed and binary candidate texts do not establish demographic bias.')
display(role_img)


## 7. Deliverable check

In [ ]:
artifacts=['gen_baseline.png','gen_prompt_grid.png','gen_img2img.png','gen_baseline_tagged.png','gen_nurse_probe.png']+[f'gallery_{i}.png' for i in range(1,5)]
for path in artifacts:
    print(('OK' if os.path.exists(path) else 'MISSING').ljust(8),path,os.path.getsize(path) if os.path.exists(path) else 0,'bytes')
assert all(os.path.exists(p) for p in artifacts)
print('All image artifacts saved. Review outputs and update the reflection with your actual observations.')


## Final reflection (draft, update after running and inspecting the generated images)

**Generation and control.** I used a fixed seed of 42 to compare guidance scales 3 and 12 and a guidance-7.5 negative-prompt variant. My initial preference is the moderate-guidance negative-prompt version because it aims to retain the warm watercolor atmosphere while discouraging blur and darkness. This is a hypothesis until the panels are inspected: negative prompts do not guarantee improved image quality, and higher guidance can exaggerate contrast or introduce artifacts.

**Editing.** I compared img2img strengths 0.35 and 0.65 against the same original reading nook. I expect 0.35 to keep the room layout more recognizable, while 0.65 should create a stronger sunset reinterpretation. The appropriate setting depends on whether preserving composition or changing the lighting is the priority. The generated comparison must be checked before asserting which edit succeeded.

**Understanding.** BLIP describes visual content in natural language, whereas CLIP ranks the image against a fixed set of candidate descriptions. I expect a reading-nook caption and strong similarity to indoor-scene or watercolor labels, but neither model can independently verify the original prompt or guarantee that every requested detail appeared. A caption might omit rain or lighting; CLIP rankings depend on the candidate set.

**Search.** The query “somewhere warm to sit and read” should retrieve the reading-nook image despite using different words from its gallery prompt. This tests semantic matching rather than exact keyword overlap. The actual ranking and cosine scores printed above are the evidence; a single successful retrieval is not a general retrieval benchmark.

**Responsible AI.** I attached AI-generated provenance metadata to the baseline PNG. Because metadata can be removed, a deployment should additionally use tamper-evident content credentials and a resilient watermark, with explicit disclosure. A neutral nurse prompt may reproduce occupational stereotypes from training data. One generated image and two CLIP descriptions are not enough to establish systematic bias, so I would test many seeds, roles, and descriptions with human review. I would also enforce consent and misuse checks to reduce impersonation, disinformation, and non-consensual image generation.